In [1]:
import numpy as np
import hnswlib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Get Sentences using Ingestion tool

In [2]:
from components import *

sentences = ingest_using_url("https://jamesclear.com/saying-no")
print(sentences)

combined = get_scores(sentences)
print(combined)
print(f'Number of scores: {len(combined['combined'])}')

features = combine_features(
    sentences=sentences,
    textrank_scores_arr=combined['textrank'],
    centroid_scores_arr=combined['centroid'],
    domain='general'
)

scores = features['combined']


c:\Users\Justin\Documents\2026 projects\python\textSummariserExample\summ\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7780.38it/s]


['The Ultimate Productivity Hack is Saying No - James Clear The Ultimate Productivity Hack is Saying No written by James Clear Decision Making Focus Life Lessons The ultimate productivity hack is saying no.', 'Not doing something will always be faster than doing it.', 'This statement reminds me of the old computer programming saying, “Remember that there is no code faster than no code.” 1 The same philosophy applies in other areas of life.', 'For example, there is no meeting that goes faster than not having a meeting at all.', "This is not to say you should never attend another meeting, but the truth is that we say yes to many things we don't actually want to do.", "There are many meetings held that don't need to be held.", 'There is a lot of code written that could be deleted.', "How often do people ask you to do something and you just reply, “Sure thing.” Three days later, you're overwhelmed by how much is on your to-do list.", 'We become frustrated by our obligations even though we 

#### HNSW = Hierarchical Navigable Small World

##### The naive approach to finding near-duplicates:
#####   Compare every sentence to every other sentence → O(N²)
#####   With 1000 sentences = 1,000,000 comparisons. Slow.

##### HNSW builds a multi-layer graph where:
#####   - Top layers : long-range connections (fast coarse navigation)
#####   - Bottom layers: short-range connections (precise local search)

##### Think of it like a road network:
#####   - Top layer   = motorways (jump far quickly)
#####   - Bottom layer = local streets (find exact destination)

##### Cost: O(N log N) — with 1000 sentences ≈ 10,000 comparisons. Much faster.
##### Accuracy: ~95-99% of exact results (the "approximate" part)

#### Key parameters:
#####   M               : connections per node (higher = more accurate, more memory)
#####   ef_construction : search width during build (higher = more accurate, slower)
#####   ef              : search width during query (higher = more accurate, slower)
#####   space='ip'      : inner product = cosine similarity on L2-normalised vectors

### 1. Vectorise

In [7]:
# 4a. VECTORISE AND L2-NORMALISE

def vectorise_sentences(sentences: list[str], bert = True) -> tuple[np.ndarray, object]:
    """
    Convert sentences to TF-IDF vectors and L2-normalise them.
 
    WHY L2-normalise?
    After normalisation every vector has magnitude = 1.
    This means: dot_product(a, b) == cosine_similarity(a, b)
    HNSW's inner product space ('ip') computes dot products, so this
    trick lets us do cosine similarity inside the HNSW index cheaply.
 
    Returns
    -------
    vectors    : np.ndarray shape (n, vocab_size), float32, unit-length
    vectorizer : fitted TfidfVectorizer (for inspection)

    """
    tfidf = None
    vectorizer = None

    if bert:
        tfidf, vectorizer = build_sbert_vectors(sentences)
    else:
        vectorizer = TfidfVectorizer(stop_words='english', min_df=1)
        try:
            tfidf = vectorizer.fit_transform(sentences).toarray().astype('float32')
        except ValueError:
            # All words are stop words — return identity matrix as fallback
            n = len(sentences)
            return np.eye(n, dtype='float32'), vectorizer
 
    # L2-normalise: divide each row vector by its magnitude
    # Before: ||v|| could be anything
    # After:  ||v|| = 1.0 for every sentence
    norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
    norms[norms == 0] = 1            # prevent divide-by-zero for blank sentences
    normalised = tfidf / norms
 
    return normalised, vectorizer

In [8]:
print("\n── 4a. TF-IDF Cosine Similarity Matrix ──")
print("   (values above threshold shown with **)\n")
vectors, _ = vectorise_sentences(sentences)
sim_matrix  = cosine_similarity(vectors)
THRESHOLD   = 0.60
 
header = "        " + "  ".join(f"S{i+1}" for i in range(len(sentences)))
print(header)
for i, row in enumerate(sim_matrix):
    vals = []
    for j, v in enumerate(row):
        cell = f"{v:.2f}"
        if v >= THRESHOLD and i != j:
            cell = f"{v:.2f}*"     # mark duplicates
        vals.append(f"{cell:>6}")
    print(f"  S{i+1}    {'  '.join(vals)}")
 
print(f"\n  * = similarity >= {THRESHOLD} (duplicate candidate)")


── 4a. TF-IDF Cosine Similarity Matrix ──
   (values above threshold shown with **)

        S1  S2  S3  S4  S5  S6  S7  S8  S9  S10  S11  S12  S13  S14  S15  S16  S17  S18  S19  S20  S21  S22  S23  S24  S25  S26  S27  S28  S29  S30  S31  S32  S33  S34  S35  S36  S37  S38  S39  S40  S41  S42  S43  S44  S45  S46  S47  S48  S49  S50  S51  S52  S53  S54  S55  S56  S57  S58  S59  S60  S61  S62  S63  S64  S65  S66  S67  S68  S69  S70  S71  S72  S73  S74  S75  S76  S77  S78  S79  S80  S81  S82  S83  S84  S85  S86  S87  S88  S89  S90  S91  S92  S93  S94
  S1      1.00    0.26    0.33    0.19    0.11    0.07    0.27    0.16    0.04    0.12    0.37    0.20    0.11    0.14    0.07    0.08    0.21    0.14    0.11    0.02    0.18    0.22    0.21    0.22    0.12    0.33    0.33    0.25    0.32    0.23    0.18    0.21    0.15    0.11    0.26    0.21    0.31    0.19    0.33    0.26    0.31    0.52    0.18    0.32    0.10    0.31    0.34    0.25    0.16    0.10    0.31    0.29    0.18    0.37    0.30

### 2. HNSW index (replace with our own later?)

In [9]:
# 4b. BUILD THE HNSW INDEX

def build_hnsw_index(vectors: np.ndarray,
                     M: int = 16,
                     ef_construction: int = 100) -> hnswlib.Index:
    """
    Build an HNSW index from L2-normalised sentence vectors.
 
    Parameters
    ----------
    vectors         : unit-length vectors, shape (n, dim)
    M               : max bi-directional connections per node
                        8-16 = fast, lower memory usage
                        32-64 = more accurate, higher memory
    ef_construction : candidate list size during graph construction
                        100 is a safe, well-tested default
 
    Returns
    -------
    index : hnswlib.Index, ready to be queried
    """
    n, dim = vectors.shape
 
    # Initialise: inner product space on vectors of size `dim`
    index = hnswlib.Index(space='ip', dim=dim)
 
    # Allocate memory and build the layered graph
    index.init_index(
        max_elements=n,
        ef_construction=ef_construction,
        M=M
    )
 
    # Add all sentence vectors; IDs are their positions (0, 1, 2, ...)
    index.add_items(vectors, ids=list(range(n)))
 
    # Set query-time search width (can be adjusted after building)
    index.set_ef(max(50, M * 2))
 
    return index

In [10]:
print("\n── 4b. Building HNSW Index ──")
n, dim = vectors.shape
index = build_hnsw_index(vectors, M=16, ef_construction=100)
print(f"  Sentences indexed : {index.element_count}")
print(f"  Vector dimensions : {dim}  (TF-IDF vocab size)")
print(f"  Connections/node  : M=16")
print(f"  Index space       : inner product (cosine on L2-normed vecs)")


── 4b. Building HNSW Index ──
  Sentences indexed : 94
  Vector dimensions : 384  (TF-IDF vocab size)
  Connections/node  : M=16
  Index space       : inner product (cosine on L2-normed vecs)


### 3. Query

In [11]:
# 4c. QUERY FOR NEAR-NEIGHBOURS

def find_near_duplicates_hnsw(vectors: np.ndarray,
                               index: hnswlib.Index,
                               threshold: float = 0.70,
                               k_neighbours: int = 5
                               ) -> list[tuple[int, int, float]]:
    """
    For each sentence, find its k nearest neighbours in the HNSW index
    and collect pairs whose similarity exceeds the threshold.
 
    Parameters
    ----------
    vectors      : L2-normalised sentence vectors
    index        : built HNSW index
    threshold    : cosine similarity cutoff — pairs above this are duplicates
                   For TF-IDF: 0.60-0.75 recommended
                   For SBERT : 0.85-0.92 recommended
    k_neighbours : how many neighbours to retrieve per sentence
 
    Returns
    -------
    duplicate_pairs : list of (i, j, similarity) tuples where i < j
    """
    n = len(vectors)
    duplicate_pairs = []
    seen_pairs = set()               # track (i,j) to avoid (j,i) duplicates
 
    for i in range(n):
        # knn_query returns ([labels], [distances]) each shape (1, k)
        labels, distances = index.knn_query(vectors[i], k=min(k_neighbours + 1, n))
        labels    = labels[0]        # flatten: shape (k,)
        distances = distances[0]     # flatten: shape (k,)
 
        for j, sim in zip(labels, distances):
            j   = int(j)
            sim = float(sim)
 
            if j == i:               # skip self-match
                continue
 
            pair = (min(i, j), max(i, j))
            if pair in seen_pairs:   # skip (j,i) if we saw (i,j)
                continue
            seen_pairs.add(pair)
 
            if sim >= threshold:
                duplicate_pairs.append((i, j, sim))
 
    return duplicate_pairs


def find_near_duplicates_exact(vectors: np.ndarray,
                                threshold: float = 0.70
                                ) -> list[tuple[int, int, float]]:
    """
    Brute-force O(N²) exact cosine similarity.
    Used for small documents (< ~50 sentences) or to verify HNSW results.
 
    Returns
    -------
    duplicate_pairs : list of (i, j, similarity) where i < j
    """
    sim_matrix = cosine_similarity(vectors)
    n = vectors.shape[0]
    pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            sim = float(sim_matrix[i, j])
            if sim >= threshold:
                pairs.append((i, j, sim))
    return pairs

In [12]:
print("\n── 4c. Top Neighbours per Sentence ──")
print(f"  {'Sentence':<6}  {'Top-3 nearest neighbours (sim)'}")
print("  " + "-" * 55)
for i in range(len(sentences)):
    labels_i, dists_i = index.knn_query(vectors[i], k=min(4, len(sentences)))
    neighbours = [
        (int(j), float(d))
        for j, d in zip(labels_i[0], dists_i[0])
        if int(j) != i
    ][:3]
    parts = "  |  ".join(f"S{j+1}={d:.3f}{'*' if d>=THRESHOLD else ''}"
                          for j, d in neighbours)
    print(f"  S{i+1:<5}  {parts}")


── 4c. Top Neighbours per Sentence ──
  Sentence  Top-3 nearest neighbours (sim)
  -------------------------------------------------------
  S1      S42=0.479  |  S85=0.523  |  S91=0.525
  S2      S3=0.366  |  S73=0.458  |  S4=0.468
  S3      S2=0.366  |  S75=0.442  |  S4=0.479
  S4      S6=0.356  |  S77=0.446  |  S5=0.450
  S5      S77=0.389  |  S6=0.433  |  S4=0.450
  S6      S4=0.356  |  S5=0.433  |  S77=0.501
  S7      S3=0.706*  |  S75=0.707*  |  S1=0.730*
  S8      S28=0.557  |  S83=0.574  |  S54=0.578
  S9      S21=0.465  |  S13=0.507  |  S71=0.543
  S10     S69=0.571  |  S88=0.638*  |  S16=0.639*
  S11     S73=0.449  |  S70=0.559  |  S42=0.563
  S12     S60=0.325  |  S30=0.412  |  S24=0.434
  S13     S12=0.495  |  S9=0.507  |  S15=0.514
  S14     S29=0.415  |  S5=0.458  |  S46=0.462
  S15     S61=0.475  |  S70=0.502  |  S13=0.514
  S16     S10=0.639*  |  S15=0.639*  |  S9=0.641*
  S17     S22=0.625*  |  S38=0.639*  |  S79=0.644*
  S18     S71=0.484  |  S22=0.521  |  S73=0.530


### 4. Remove duplicates

In [13]:
# 4d. RESOLVE DUPLICATES — decide which sentence to keep

def resolve_duplicates(sentences: list[str],
                        scores: np.ndarray,
                        duplicate_pairs: list[tuple[int, int, float]],
                        verbose: bool = True
                        ) -> set[int]:
    """
    Given detected duplicate pairs, remove the lower-scoring sentence.
 
    Strategy: always keep the sentence with the higher Step 3 score.
    Process most-similar pairs first so the most obvious duplicates
    are resolved before any cascading effects.
 
    Returns
    -------
    keep : set of indices to retain (all others are duplicates)
    """
    keep = set(range(len(sentences)))
 
    # Process most-similar pairs first
    for i, j, sim in sorted(duplicate_pairs, key=lambda x: -x[2]):
        if i not in keep or j not in keep:
            continue                 # already removed by a previous resolution
 
        # Keep the higher-scoring sentence
        if scores[i] >= scores[j]:
            keep.discard(j)
            winner, loser = i, j
        else:
            keep.discard(i)
            winner, loser = j, i
 
        if verbose:
            print(f"    Duplicate (sim={sim:.3f}): removed S{loser+1} "
                  f"(score={scores[loser]:.2f}), kept S{winner+1} "
                  f"(score={scores[winner]:.2f})")
 
    return keep

In [14]:
print(f"\n── Duplicate Pairs Detected (sim ≥ {THRESHOLD}) ──")
pairs = find_near_duplicates_hnsw(vectors, index, threshold=THRESHOLD, k_neighbours=4)
if pairs:
    for i, j, sim in sorted(pairs, key=lambda x: -x[2]):
        print(f"\n  S{i+1} ↔ S{j+1}  similarity={sim:.4f}")
        print(f"    S{i+1}: {sentences[i]}")
        print(f"    S{j+1}: {sentences[j]}")
        print(f"    Decision: keep S{i+1 if scores[i] >= scores[j] else j+1} "
            f"(score={max(scores[i], scores[j]):.2f})")
else:
    print("  None found.")


── Duplicate Pairs Detected (sim ≥ 0.6) ──

  S93 ↔ S91  similarity=0.8482
    S93: The book has sold over 25 million copies worldwide and has been translated into more than 60 languages.
    S91: About the Author James Clear writes about habits, decision making, and continuous improvement.
    Decision: keep S91 (score=0.24)

  S93 ↔ S44  similarity=0.8331
    S93: The book has sold over 25 million copies worldwide and has been translated into more than 60 languages.
    S44: It means saying no to the hundred other good ideas that there are.
    Decision: keep S44 (score=0.58)

  S90 ↔ S10  similarity=0.8154
    S90: Enter your email now and join us.
    S10: 2 It's worth asking if things are necessary.
    Decision: keep S10 (score=0.51)

  S92 ↔ S93  similarity=0.8110
    S92: He is the author of the #1 New York Times bestseller, Atomic Habits .
    S93: The book has sold over 25 million copies worldwide and has been translated into more than 60 languages.
    Decision: keep S92 (s

### 5. Full

In [15]:
# FULL PIPELINE

def deduplicate(sentences: list[str],
                scores: np.ndarray,
                threshold: float = 0.70,
                hnsw_min_sentences: int = 20
                ) -> tuple[list[str], np.ndarray, list[int]]:
    """
    Full deduplication pipeline. Auto-selects exact vs HNSW based on N.
 
    Parameters
    ----------
    sentences           : cleaned sentences from Step 1
    scores              : combined scores from Step 3
    threshold           : cosine similarity cutoff (0.70 for TF-IDF)
    hnsw_min_sentences  : use HNSW when n >= this, exact otherwise
 
    Returns
    -------
    deduped_sentences : remaining sentences after deduplication
    deduped_scores    : their corresponding scores
    kept_indices      : original indices that survived
    """
    n = len(sentences)
    if n <= 1:
        return sentences, scores, list(range(n))
 
    # 4a: vectorise
    vectors, _ = vectorise_sentences(sentences)
 
    # 4b + 4c: find duplicates
    if n >= hnsw_min_sentences:
        print(f"  Method: HNSW  (n={n})")
        index = build_hnsw_index(vectors)
        pairs = find_near_duplicates_hnsw(vectors, index, threshold)
    else:
        print(f"  Method: Exact cosine  (n={n})")
        pairs = find_near_duplicates_exact(vectors, threshold)
 
    print(f"  Found {len(pairs)} duplicate pair(s) at threshold={threshold}")
 
    # 4d: resolve
    keep_set     = resolve_duplicates(sentences, scores, pairs)
    kept_indices = sorted(keep_set)
 
    # 4e: build output
    return (
        [sentences[i] for i in kept_indices],
        scores[np.array(kept_indices)],
        kept_indices
    )

In [16]:
print(f"\n── 4d+4e. Running Full Deduplication ──")
deduped_sents, deduped_scores, kept_idx = deduplicate(
    sentences, scores, threshold=THRESHOLD, hnsw_min_sentences=20
)
 
print(f"\n── Results ──")
print(f"  Before   : {len(sentences)} sentences")
print(f"  After    : {len(deduped_sents)} sentences")
print(f"  Removed  : {len(sentences) - len(deduped_sents)} duplicate(s)")
print(f"  Kept idx : {kept_idx}")
 
print(f"\n── Kept Sentences (pass these to Step 5) ──")
for sent, score in zip(deduped_sents, deduped_scores):
    print(f"  score={score:.2f}  {sent}")


── 4d+4e. Running Full Deduplication ──
  Method: HNSW  (n=94)
  Found 105 duplicate pair(s) at threshold=0.6
    Duplicate (sim=0.869): removed S93 (score=0.00), kept S7 (score=0.29)
    Duplicate (sim=0.818): removed S90 (score=0.07), kept S19 (score=0.36)
    Duplicate (sim=0.812): removed S89 (score=0.08), kept S92 (score=0.10)
    Duplicate (sim=0.806): removed S92 (score=0.10), kept S84 (score=0.64)
    Duplicate (sim=0.784): removed S7 (score=0.29), kept S6 (score=0.55)
    Duplicate (sim=0.747): removed S78 (score=0.25), kept S4 (score=0.52)
    Duplicate (sim=0.741): removed S19 (score=0.36), kept S29 (score=0.84)
    Duplicate (sim=0.741): removed S76 (score=0.44), kept S70 (score=0.59)
    Duplicate (sim=0.741): removed S45 (score=0.31), kept S54 (score=0.58)
    Duplicate (sim=0.705): removed S87 (score=0.18), kept S8 (score=0.64)
    Duplicate (sim=0.697): removed S74 (score=0.23), kept S66 (score=0.27)
    Duplicate (sim=0.692): removed S16 (score=0.46), kept S13 (score=

In [17]:
print(f"\n── Effect of Similarity Threshold on Deduplication ──")
print(f"  {'Threshold':>10}  {'Pairs Found':>12}  {'Removed':>8}  {'Kept':>6}")
print("  " + "-" * 42)
for t in [0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    p = find_near_duplicates_exact(vectors, threshold=t)
    k = resolve_duplicates(sentences, scores, p, verbose=False)
    removed = len(sentences) - len(k)
    print(f"  {t:>10.2f}  {len(p):>12}  {removed:>8}  {len(k):>6}")


── Effect of Similarity Threshold on Deduplication ──
   Threshold   Pairs Found   Removed    Kept
  ------------------------------------------
        0.40           505        61      33
        0.50           165        40      54
        0.60            29        15      79
        0.70             6         6      88
        0.80             1         1      93
        0.90             1         1      93


In [18]:
print(f"\n── HNSW vs Exact: Result Comparison (threshold={THRESHOLD}) ──")
exact_pairs = set((i, j) for i, j, _ in find_near_duplicates_exact(vectors, THRESHOLD))
hnsw_pairs  = set((i, j) for i, j, _ in find_near_duplicates_hnsw(vectors, index, THRESHOLD))
print(f"  Exact found : {len(exact_pairs)} pairs")
print(f"  HNSW found  : {len(hnsw_pairs)} pairs")
missed = exact_pairs - hnsw_pairs
print(f"  HNSW missed : {len(missed)} pairs  {'(none — perfect match!)' if not missed else missed}")


── HNSW vs Exact: Result Comparison (threshold=0.6) ──
  Exact found : 29 pairs
  HNSW found  : 105 pairs
  HNSW missed : 29 pairs  {(43, 46), (22, 23), (4, 76), (40, 41), (29, 47), (40, 47), (38, 59), (28, 45), (26, 29), (50, 57), (20, 22), (11, 59), (1, 2), (34, 36), (28, 29), (34, 45), (30, 32), (28, 38), (56, 57), (50, 56), (3, 5), (25, 45), (38, 45), (46, 58), (40, 54), (34, 59), (27, 29), (25, 26), (45, 59)}
